In [1]:
!pip install opencv-python mediapipe pygame

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.6 MB 279.8 kB/s eta 0:00:37
   - -------------------------------------- 0.5/10.6 MB 279.8 kB/s eta 0:00:37
   - -------------------------------------- 0.5/10.6 MB 279.8 kB/s eta 0:00:37
   - -------------------------------------- 0.5/10.6 MB 279.8 kB/s eta 0:00:37
   - -------------------------------------- 0.5/10.6 MB 279.8 kB/s eta 0:00:37
   - -------------------------------------- 0

In [56]:
import cv2
import mediapipe as mp
import pygame
import os
import numpy as np
import time
import random
import math

# ==========================================
#  超級設定區 (請在這裡調整大小與靈敏度)
# ==========================================

# 1. 遊戲視窗設定
WINDOW_WIDTH = 640
WINDOW_HEIGHT = 480
ENABLE_SEGMENTATION = True  # 電腦卡頓請改成 False

# 2. 濾鏡大小 (AR Filter Size) - [已拆分]
# --- A. 秋季關卡 (松鼠) - 這是您剛剛測試好的數值 ---
AUTUMN_HAND_SIZE = 200      # 手部大小
AUTUMN_FACE_W = 450         # 臉部寬度
AUTUMN_FACE_H = 450         # 臉部高度

# --- B. TSMC 關卡 (工程師) - 請在這裡單獨調整 TSMC ---
TSMC_HAND_SIZE = 220        # TSMC 手套大小 (通常手套大一點比較可愛)
TSMC_FACE_W = 280           # TSMC 無塵衣寬度 (您可以試著調大或調小)
TSMC_FACE_H = 420           # TSMC 無塵衣高度

# 3. 濾鏡位置微調 (已拆分模式)
# 定義不同關卡的偏移量設定
FILTER_CONFIGS = {
    # === 🐿️ 模式 1: 松鼠 (Squirrel) ===
    "SQUIRREL": {
        "face_offset_x": 110,   # 松鼠耳朵可能需要左右微調
        "face_offset_y": 50,    # 松鼠耳朵在頭頂
        "hand_offset_x": 0,
        "hand_offset_y": 0,
        "scale_face": 1.0       # 臉部貼圖縮放 (預設)
    },
    
    # === 🏭 模式 2: 台積電 (TSMC) ===
    "TSMC": {
        "face_offset_x": 0,     # 頭盔通常需要置中，X 偏移可能要歸 0
        "face_offset_y": 70,   # 頭盔要包住臉，Y 可能不需要往下移太多
        "hand_offset_x": 0,
        "hand_offset_y": 0,
        "scale_face": 1.2       # 頭盔可能需要放大一點才蓋得住頭
    }
}

# 4. 結算畫面分數顯示微調
SCORE_SCALE = 0.30          # 結算數字大小
SCORE_OFFSET_X = 0          # 結算數字左右偏移

# 5. 遊戲中 UI 位置微調
UI_TEXT_Y = 35              # 遊戲中分數與時間的高度
UI_TEXT_SIZE = 0.2          # 遊戲中數字的大小

# ==========================================

class ResourceManager:
    def __init__(self, asset_folder='assets'):
        self.folder = asset_folder
        self.images = {}
        self.sounds = {}
        try:
            pygame.mixer.init()
        except Exception as e:
            print(f"音效系統初始化失敗: {e}")

    def load_image(self, name):
        if name in self.images: return self.images[name]
        path = os.path.join(self.folder, name)
        if not os.path.exists(path): return None
        try:
            img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
            self.images[name] = img
            return img
        except Exception as e:
            print(f"讀取錯誤 {name}: {e}")
            return None

    def play_music(self, name, loops=-1):
        path = os.path.join(self.folder, name)
        if os.path.exists(path):
            try:
                pygame.mixer.music.stop()
                pygame.mixer.music.load(path)
                pygame.mixer.music.play(loops)
            except: pass

    def play_sound(self, name):
        if name not in self.sounds:
            path = os.path.join(self.folder, name)
            if os.path.exists(path):
                try: self.sounds[name] = pygame.mixer.Sound(path)
                except: return
            else: return
        if name in self.sounds: self.sounds[name].play()

    def stop_music(self):
        pygame.mixer.music.stop()

def overlay_image(bg, fg, x, y, w=None, h=None):
    if fg is None: return bg
    if w is not None and h is not None:
        fg = cv2.resize(fg, (w, h))
    
    fh, fw = fg.shape[:2]
    bh, bw = bg.shape[:2]
    
    if x >= bw or y >= bh: return bg
    
    crop_x = max(0, x)
    crop_y = max(0, y)
    crop_w = min(bw - crop_x, fw - (crop_x - x))
    crop_h = min(bh - crop_y, fh - (crop_y - y))
    
    if crop_w <= 0 or crop_h <= 0: return bg

    fg_x = crop_x - x
    fg_y = crop_y - y
    fg_crop = fg[fg_y:fg_y+crop_h, fg_x:fg_x+crop_w]
    bg_crop = bg[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w]

    if fg_crop.shape[2] == 4:
        alpha = fg_crop[:, :, 3] / 255.0
        for c in range(3):
            bg_crop[:, :, c] = (1.0 - alpha) * bg_crop[:, :, c] + alpha * fg_crop[:, :, c]
    else:
        bg_crop[:] = fg_crop

    bg[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w] = bg_crop
    return bg

def draw_number(bg, number, res_mgr, center_x, y, scale=1.0):
    """改良版數字繪製，以 center_x 為中心"""
    str_num = str(int(number))
    
    # 1. 先計算總寬度
    total_width = 0
    img_list = []
    
    for char in str_num:
        img_name = f"num_{char}.png"
        img = res_mgr.load_image(img_name)
        if img is not None:
            w = int(img.shape[1] * scale)
            h = int(img.shape[0] * scale)
            img_list.append((img, w, h))
            total_width += w
        else:
            # 沒圖的 fallback
            img_list.append((char, 20 * scale, 30)) 
            total_width += 20 * scale

    # 2. 計算起始 X，讓整體置中
    current_x = int(center_x - (total_width / 2))

    # 3. 繪製
    for item in img_list:
        if isinstance(item[0], str): # 文字模式
            char, w, h = item
            cv2.putText(bg, char, (current_x, y + 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
            current_x += int(w)
        else: # 圖片模式
            img, w, h = item
            bg = overlay_image(bg, img, current_x, y, w, h)
            current_x += w
            
    return bg

class GameObject:
    def __init__(self, obj_type, img_name, start_x, start_y, w, h):
        self.type = obj_type
        self.img_name = img_name
        self.x = start_x
        self.y = start_y
        self.w = w
        self.h = h
        self.state = 'falling'
        self.vx = 0
        self.vy = 5
        self.holder_id = None

class ARGame:
    def __init__(self):
        self.res = ResourceManager()
        self.cap = cv2.VideoCapture(0)
        self.cap.set(3, WINDOW_WIDTH)
        self.cap.set(4, WINDOW_HEIGHT)
        
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(model_complexity=0, max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
        
        self.mp_face = mp.solutions.face_mesh
        self.face_mesh = self.mp_face.FaceMesh(max_num_faces=1, refine_landmarks=False)
        self.mp_draw = mp.solutions.drawing_utils
        
        if ENABLE_SEGMENTATION:
            self.mp_selfie = mp.solutions.selfie_segmentation
            self.segmentation = self.mp_selfie.SelfieSegmentation(model_selection=0)

        self.STATE_MENU = 0
        self.STATE_TUTORIAL = 1
        self.STATE_GAME = 2
        self.STATE_GAMEOVER = 3
        
        self.current_state = self.STATE_MENU
        self.current_level = None 
        
        self.score = 0
        self.start_time = 0
        self.game_duration = 60
        self.objects = []
        self.static_objects = []
        self.last_spawn_time = 0
        
        self.hover_btn_name = None
        self.hover_start_time = 0
        self.prev_hand_x = {}

    def switch_state(self, new_state):
        self.current_state = new_state
        self.hover_btn_name = None
        self.hover_start_time = 0

    def get_hand_status(self, landmarks):
        wrist = landmarks[0]
        tips = [8, 12, 16, 20]
        pips = [6, 10, 14, 18]
        folded_count = 0
        for i in range(4):
            dist_tip = math.hypot(landmarks[tips[i]].x - wrist.x, landmarks[tips[i]].y - wrist.y)
            dist_pip = math.hypot(landmarks[pips[i]].x - wrist.x, landmarks[pips[i]].y - wrist.y)
            if dist_tip < dist_pip:
                folded_count += 1
        if folded_count >= 3: return "FIST"
        else: return "OPEN"

    def check_hover(self, cx, cy, rect, name, frame):
        x, y, w, h = rect
        # 檢查手指是否在按鈕範圍內
        if x < cx < x + w and y < cy < y + h:
            # 如果是剛移入新的按鈕，重置計時器
            if self.hover_btn_name != name:
                self.hover_btn_name = name
                self.hover_start_time = time.time()
            
            # 計算經過時間與進度 (0.0 ~ 1.0)
            elapsed = time.time() - self.hover_start_time
            progress = min(elapsed / 3.0, 1.0) # 確保不超過 1.0
            
            # --- ✨ 修改開始：圓形半透明讀取條 ✨ ---
            
            # 1. 建立一個圖層 (Overlay) 用來畫半透明圖形
            overlay = frame.copy()
            
            # 設定圓的參數
            radius = 30         # 圓的半徑 (可調整大小)
            center = (cx, cy)   # 圓心跟隨手指位置
            color = (255, 255, 255) # 白色
            
            # 2. 畫底圓 (淡淡的軌道，非必要，但比較好看)
            cv2.circle(overlay, center, radius, color, 2)
            
            # 3. 畫進度扇形 (實心)
            # 參數說明：(影像, 圓心, 軸長, 旋轉角, 起始角, 結束角, 顏色, 粗細)
            # -90 度是為了讓它從「12點鐘方向」開始轉
            angle_end = 360 * progress
            cv2.ellipse(overlay, center, (radius, radius), -90, 0, angle_end, color, -1)
            
            # 4. 混合圖層 (產生半透明效果)
            # alpha=0.4 代表白色圓形只有 40% 不透明度
            cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
            
            # --- ✨ 修改結束 ✨ ---

            # 觸發判定
            if elapsed >= 3.0:
                self.hover_btn_name = None 
                return True
        else:
            # 手指離開按鈕，重置狀態
            if self.hover_btn_name == name:
                self.hover_btn_name = None
        return False

    def spawn_logic(self, time_left):
        # === 修改點 1: 縮短生成時間 (Interval) ===
        if time_left > 40:
            interval = 4.0
        elif time_left > 20: 
            interval = 3.0
        else: 
            interval = 2.0
        
        if time.time() - self.last_spawn_time > interval:
            self.last_spawn_time = time.time()
            
            # === 修改點 2: 一次掉落的數量 (Batch) ===
            # 隨機決定這次要掉 1 個還是 2 個，讓遊戲更有變化
            spawn_count = random.randint(1, 2) 

            for _ in range(spawn_count):
                spawn_x = random.randint(50, WINDOW_WIDTH - 100)
                
                if self.current_level == 'autumn':
                    choices = ['pinecone', 'flowercat', 'orangecat', 'tigercat', 'spider']
                else:
                    choices = ['wafer', 'greensnack', 'yellowsnack', 'bug']
                
                # 隨機選一個物品
                choice = random.choice(choices)
                
                # 建立物件
                obj = GameObject(choice, f"{choice}.png", spawn_x, -80, 80, 80)
                
                # 微調：如果是蜘蛛或Bug，讓它掉得比普通物品更快一點
                if choice in ['spider', 'bug']:
                    obj.vy = 8  # 壞東西掉比較快 (原本是 5)
                else:
                    obj.vy = random.randint(5, 7) # 好東西速度在 5~7 之間浮動
                
                self.objects.append(obj)
    # 這是新增的函式：用來畫半透明的準心

    def draw_cursor(self, img, x, y):
        # 1. 建立一個透明圖層
        overlay = img.copy()
        
        # 2. 設定顏色 (白色)
        color = (255, 255, 255)
        
        # 3. 畫中心實心點 (半徑 5)
        cv2.circle(overlay, (x, y), 5, color, -1)
        
        # 4. 畫外圍空心圓 (半徑 15, 線條粗細 2)
        cv2.circle(overlay, (x, y), 15, color, 2)
        
        # 5. 混合圖層 (透明度 0.5)
        # 參數: (圖層, 透明度, 原圖, 透明度, gamma)
        cv2.addWeighted(overlay, 0.5, img, 0.5, 0, img)
        
        return img

    def update_menu(self, frame):
        bg = self.res.load_image('bg_start.png')
        display = cv2.resize(bg, (WINDOW_WIDTH, WINDOW_HEIGHT)) if bg is not None else frame.copy()

        rect_a = (50, 300, 200, 100)
        rect_t = (390, 300, 200, 100)
        display = overlay_image(display, self.res.load_image('btn_autumn.png'), *rect_a)
        display = overlay_image(display, self.res.load_image('btn_tsmc.png'), *rect_t)
        
        if self.hand_results.multi_hand_landmarks:
            lm = self.hand_results.multi_hand_landmarks[0]
            cx, cy = int(lm.landmark[9].x * WINDOW_WIDTH), int(lm.landmark[9].y * WINDOW_HEIGHT)
            display = self.draw_cursor(display, cx, cy)
            
            if self.check_hover(cx, cy, rect_a, 'autumn', display):
                self.current_level = 'autumn'
                self.switch_state(self.STATE_TUTORIAL)
                self.res.play_music('music_menu.mp3')
            elif self.check_hover(cx, cy, rect_t, 'tsmc', display):
                self.current_level = 'tsmc'
                self.switch_state(self.STATE_TUTORIAL)
                self.res.play_music('music_menu.mp3')
        return display

    def update_tutorial(self, frame):
        t_name = 'tutorial_autumn.png' if self.current_level == 'autumn' else 'tutorial_tsmc.png'
        bg = self.res.load_image(t_name)
        display = cv2.resize(bg, (WINDOW_WIDTH, WINDOW_HEIGHT)) if bg is not None else frame.copy()

        rect_s = (WINDOW_WIDTH - 220, WINDOW_HEIGHT - 120, 200, 100)
        display = overlay_image(display, self.res.load_image('btn_start.png'), *rect_s)
        
        if self.hand_results.multi_hand_landmarks:
            lm = self.hand_results.multi_hand_landmarks[0]
            cx, cy = int(lm.landmark[9].x * WINDOW_WIDTH), int(lm.landmark[9].y * WINDOW_HEIGHT)
            display = self.draw_cursor(display, cx, cy)
            
            if self.check_hover(cx, cy, rect_s, 'start', display):
                self.switch_state(self.STATE_GAME)
                self.start_time = time.time()
                self.score = 0
                self.objects = []
                self.static_objects = []
                m_name = 'music_autumn.mp3' if self.current_level == 'autumn' else 'music_tsmc.mp3'
                self.res.play_music(m_name)
        return display

    def update_game(self, frame):
        display = frame.copy()
        if ENABLE_SEGMENTATION:
            try:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = self.segmentation.process(rgb)
                condition = np.stack((results.segmentation_mask,) * 3, axis=-1) > 0.5
                bg_name = 'bg_autumn.png' if self.current_level == 'autumn' else 'bg_tsmc.png'
                bg = self.res.load_image(bg_name)
                bg = cv2.resize(bg, (WINDOW_WIDTH, WINDOW_HEIGHT)) if bg is not None else np.zeros_like(frame)
                display = np.where(condition, frame, bg)
            except: pass
        else:
            bg_name = 'bg_autumn.png' if self.current_level == 'autumn' else 'bg_tsmc.png'
            bg = self.res.load_image(bg_name)
            if bg is not None:
                bg = cv2.resize(bg, (WINDOW_WIDTH, WINDOW_HEIGHT))
                display = cv2.addWeighted(bg, 0.7, frame, 0.3, 0)
        
        elapsed = time.time() - self.start_time
        time_left = max(0, self.game_duration - int(elapsed))
        
        if time_left == 0:
            self.switch_state(self.STATE_GAMEOVER)
            self.gameover_start_time = time.time()
            return display

        self.spawn_logic(time_left)

        hands_data = []
        if self.hand_results.multi_hand_landmarks:
            for i, h_lm in enumerate(self.hand_results.multi_hand_landmarks):
                # 繪製骨架
                self.mp_draw.draw_landmarks(display, h_lm, self.mp_hands.HAND_CONNECTIONS)
                
                cx, cy = int(h_lm.landmark[9].x * WINDOW_WIDTH), int(h_lm.landmark[9].y * WINDOW_HEIGHT)
                status = self.get_hand_status(h_lm.landmark)
                #cv2.putText(display, status, (cx, cy-20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
                display = self.draw_cursor(display, cx, cy)
                
                dx = cx - self.prev_hand_x.get(i, cx)
                self.prev_hand_x[i] = cx
                hands_data.append({'id': i, 'x': cx, 'y': cy, 'status': status, 'dx': dx})
                
                # 手部濾鏡 (含左右判定 + 位置偏移)
                hand_label = self.hand_results.multi_handedness[i].classification[0].label # 'Left' or 'Right'
                level_prefix = 'squirrel' if self.current_level == 'autumn' else 'tsmc'
                hand_side = hand_label.lower()
                f_hand = f'filter_{level_prefix}hand_{hand_side}.png'
                
                # [邏輯更新] 判斷是哪個關卡，使用對應的大小變數
                if self.current_level == 'autumn':
                    current_hand_size = AUTUMN_HAND_SIZE
                else:
                    current_hand_size = TSMC_HAND_SIZE

                # 應用 OFFSET
                hx = cx - current_hand_size//2 + HAND_OFFSET_X
                hy = cy - current_hand_size//2 + HAND_OFFSET_Y
                display = overlay_image(display, self.res.load_image(f_hand), hx, hy, current_hand_size, current_hand_size)

        if self.face_results.multi_face_landmarks:
            f_lm = self.face_results.multi_face_landmarks[0]
            nx, ny = int(f_lm.landmark[4].x * WINDOW_WIDTH), int(f_lm.landmark[4].y * WINDOW_HEIGHT)
            
            # 臉部濾鏡 (含位置偏移)
            f_face = 'filter_squirrel.png' if self.current_level == 'autumn' else 'filter_suit.png'
            
            # [邏輯更新] 判斷是哪個關卡，使用對應的臉部大小
            if self.current_level == 'autumn':
                current_face_w = AUTUMN_FACE_W
                current_face_h = AUTUMN_FACE_H
            else:
                current_face_w = TSMC_FACE_W
                current_face_h = TSMC_FACE_H

            # 應用 OFFSET
            # 1. 判斷現在要讀取哪一組設定
            mode_key = "SQUIRREL" if self.current_level == 'autumn' else "TSMC"
            cfg = FILTER_CONFIGS[mode_key]

            # 2. 從字典取出對應的偏移量 (如果字典沒寫，就預設為 0)
            off_x = cfg.get("face_offset_x", 0)
            off_y = cfg.get("face_offset_y", 0)

            # 3. 計算座標 (臉中心 - 圖片一半 + 專屬偏移量)
            fx = nx - current_face_w // 2 + off_x
            fy = ny - current_face_h // 2 + off_y
            
            display = overlay_image(display, self.res.load_image(f_face), fx, fy, current_face_w, current_face_h)

        for obj in self.static_objects:
            display = overlay_image(display, self.res.load_image(obj.img_name), int(obj.x), int(obj.y), obj.w, obj.h)
            
        target_rect = (WINDOW_WIDTH - 150, WINDOW_HEIGHT - 120, 120, 120) if self.current_level == 'autumn' else (WINDOW_WIDTH - 120, WINDOW_HEIGHT - 300, 100, 300)
        if self.current_level == 'autumn':
            display = overlay_image(display, self.res.load_image('basket.png'), *target_rect)

        active_objs = []
        for obj in self.objects:
            obj.y += obj.vy
            handled = False
            
            # A. 抓取 (改良判定)
            target_grab = ['pinecone'] if self.current_level == 'autumn' else ['greensnack']
            if obj.type in target_grab:
                if obj.state == 'falling':
                    for hand in hands_data:
                        if math.hypot(hand['x'] - obj.x, hand['y'] - obj.y) < 70 and hand['status'] == 'FIST':
                            obj.state = 'held'; obj.holder_id = hand['id']
                elif obj.state == 'held':
                    holder = next((h for h in hands_data if h['id'] == obj.holder_id), None)
                    if holder:
                        obj.x, obj.y = holder['x'], holder['y']
                        tx, ty, tw, th = (WINDOW_WIDTH - 150, WINDOW_HEIGHT - 120, 150, 120) if self.current_level=='autumn' else (50, WINDOW_HEIGHT-150, 400, 150)
                        if tx < obj.x < tx+tw and ty < obj.y < ty+th and holder['status'] == 'OPEN':
                            self.score += 1
                            self.res.play_sound('sound_drop_pinecones.mp3' if obj.type=='pinecone' else 'sound_dingdong.mp3')
                            obj.state = 'placed'; obj.y = ty + th - 30 - len(self.static_objects)*5
                            self.static_objects.append(obj); handled = True
                        elif holder['status'] == 'OPEN': obj.state = 'falling'
                    else: obj.state = 'falling'

            # B. 雙手接
            target_catch = ['flowercat', 'orangecat', 'tigercat'] if self.current_level == 'autumn' else ['wafer']
            if obj.type in target_catch and obj.state == 'falling':
                if len(hands_data) == 2 and all(h['status'] == 'OPEN' for h in hands_data):
                    if all(abs(h['y'] - obj.y) < 80 for h in hands_data): obj.state = 'held_two'
            elif obj.state == 'held_two':
                if len(hands_data) == 2:
                    obj.x, obj.y = (hands_data[0]['x'] + hands_data[1]['x']) // 2, (hands_data[0]['y'] + hands_data[1]['y']) // 2
                    if obj.y > WINDOW_HEIGHT * 0.8:
                        self.score += 1
                        self.res.play_sound('sound_meow.mp3' if 'cat' in obj.type else 'sound_drop_wafer.mp3')
                        obj.state = 'placed'; self.static_objects.append(obj); handled = True
                else: obj.state = 'falling'

            # C. 揮動
            target_swipe = ['spider'] if self.current_level == 'autumn' else ['yellowsnack', 'bug']
            if obj.type in target_swipe and obj.state == 'falling':
                for hand in hands_data:
                    if math.hypot(hand['x'] - obj.x, hand['y'] - obj.y) < 80 and abs(hand['dx']) > 15:
                        self.score += 1; self.res.play_sound('sound_wave.mp3')
                        obj.state = 'flying_out'; obj.vx = 20 if hand['dx'] > 0 else -20; obj.vy = -5
            if obj.state == 'flying_out':
                obj.x += obj.vx; obj.y += obj.vy
                if obj.x < -100 or obj.x > WINDOW_WIDTH + 100: handled = True

            if not handled:
                display = overlay_image(display, self.res.load_image(obj.img_name), int(obj.x), int(obj.y), obj.w, obj.h)
                if obj.y < WINDOW_HEIGHT: active_objs.append(obj)
        
        self.objects = active_objs
        
        # 分數板與 UI (保持您設定的 110/50 邏輯)
        display = overlay_image(display, self.res.load_image('btn_score.png'), 20, 20, 120, 50)
        display = draw_number(display, self.score, self.res, 110, UI_TEXT_Y, UI_TEXT_SIZE) 
        
        display = overlay_image(display, self.res.load_image('btn_time.png'), WINDOW_WIDTH-140, 20, 120, 50)
        display = draw_number(display, time_left, self.res, WINDOW_WIDTH-50, UI_TEXT_Y, UI_TEXT_SIZE)
        
        return display

    def update_gameover(self, frame):
        display = frame.copy()
        grey = self.res.load_image('bg_grey.png')
        display = cv2.addWeighted(display, 0.3, cv2.resize(grey, (WINDOW_WIDTH, WINDOW_HEIGHT)) if grey is not None else np.zeros_like(frame), 0.7, 0)
            
        elapsed = time.time() - self.gameover_start_time
        if elapsed < 1.5:
            t_up = self.res.load_image('text_timesup.png')
            display = overlay_image(display, t_up, WINDOW_WIDTH//2-100, WINDOW_HEIGHT//2-50, 200, 100)
            self.res.stop_music()
        else:
            t_score = self.res.load_image('text_finalscore.png')
            display = overlay_image(display, t_score, WINDOW_WIDTH//2-150, 100, 300, 80)
            
            # 使用設定區的參數
            center_x = (WINDOW_WIDTH // 2) + SCORE_OFFSET_X
            display = draw_number(display, self.score, self.res, center_x, 118, SCORE_SCALE)
            
            btns = [
                ('btn_restart.png', 'restart', (WINDOW_WIDTH//2-75, 180, 150, 60)),
                ('btn_menu.png', 'menu', (WINDOW_WIDTH//2-75, 280, 150, 60)),
                ('btn_quit.png', 'quit', (WINDOW_WIDTH//2-75, 380, 150, 60))
            ]
            
            for name, action, rect in btns:
                display = overlay_image(display, self.res.load_image(name), *rect)
                if self.hand_results.multi_hand_landmarks:
                    lm = self.hand_results.multi_hand_landmarks[0]
                    cx, cy = int(lm.landmark[9].x * WINDOW_WIDTH), int(lm.landmark[9].y * WINDOW_HEIGHT)
                    #cv2.circle(display, (cx, cy), 10, (0, 255, 255), -1)
                    display = self.draw_cursor(display, cx, cy)
                    
                    if self.check_hover(cx, cy, rect, action, display):
                        if action == 'restart':
                            self.switch_state(self.STATE_GAME)
                            self.start_time = time.time()
                            self.score = 0
                            self.objects = []
                            self.static_objects = []
                            m_name = 'music_autumn.mp3' if self.current_level == 'autumn' else 'music_tsmc.mp3'
                            self.res.play_music(m_name)
                        elif action == 'menu':
                            self.switch_state(self.STATE_MENU)
                            self.res.play_music('music_menu.mp3')
                        elif action == 'quit':
                            return None 
        return display

    def run(self):
        print("遊戲啟動...按 'q' 可強制退出")
        self.res.play_music('music_menu.mp3')
        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            
            self.hand_results = self.hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            self.face_results = self.face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            
            if self.current_state == self.STATE_MENU: final_frame = self.update_menu(frame)
            elif self.current_state == self.STATE_TUTORIAL: final_frame = self.update_tutorial(frame)
            elif self.current_state == self.STATE_GAME: final_frame = self.update_game(frame)
            elif self.current_state == self.STATE_GAMEOVER:
                final_frame = self.update_gameover(frame)
                if final_frame is None: break
            
            cv2.imshow("AR Game Project", final_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
        self.cap.release()
        cv2.destroyAllWindows()
        pygame.mixer.quit()

if __name__ == "__main__":
    game = ARGame()
    game.run()

遊戲啟動...按 'q' 可強制退出
